<a href="https://colab.research.google.com/github/shreyas20240/ShreyasYadav_25SCS1003001332_IILM-GN/blob/main/ImageGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import zipfile
import os

In [ ]:
# Upload dataset.zip from your computer
print("Please upload your dataset.zip file...")
uploaded = files.upload()   # select dataset.zip

# If your file is named something else, this will detect it
zip_name = "dataset.zip"
if zip_name not in uploaded:
    zip_name = list(uploaded.keys())[0]
    print("Using uploaded file:", zip_name)

# Unzip into /content/dataset
extract_path = "/content/dataset"
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped to:", extract_path)
print("Folders inside dataset:", os.listdir(extract_path))

Please upload your dataset.zip file...


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import shutil, os

# Remove __MACOSX if exists
shutil.rmtree("dataset/__MACOSX", ignore_errors=True)

# Move inner dataset/* up one level if nested
inner_path = "dataset/dataset"
if os.path.exists(inner_path):
    for item in os.listdir(inner_path):
        src_path = os.path.join(inner_path, item)
        dst_path = os.path.join("dataset/", item)

        # If the destination path already exists, remove it to allow the move
        if os.path.exists(dst_path):
            if os.path.isdir(dst_path):
                shutil.rmtree(dst_path) # Remove existing directory
            else:
                os.remove(dst_path) # Remove existing file

        shutil.move(src_path, "dataset/")
    shutil.rmtree(inner_path)

print("✅ Final structure:", os.listdir("dataset"))

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import numpy as np
import random

IMG_SIZE = 224
BATCH_SIZE = 32

# If output of previous cell shows one more subfolder,
# change path accordingly. Usually this is correct:
data_dir = "/content/dataset"

datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2   # 20% for validation
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

num_classes = train_generator.num_classes
class_indices = train_generator.class_indices       # {"Aeroplane":0, ...}
idx_to_class = {v: k for k, v in class_indices.items()}
class_names = [idx_to_class[i].strip() for i in range(num_classes)] # Strip whitespace

print("Number of classes:", num_classes)
print("Class names:", class_names)

In [ ]:
# Pretrained MobileNetV2 backbone
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model
base_model.trainable = False

# Add our classifier on top
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS = 10  # increase later if you want

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

# Optional: save model
model.save("image_classifier_mobilenetv2.h5")
print("Model saved as image_classifier_mobilenetv2.h5")

In [ ]:
# Install and load a caption model locally (runs in-colab, no external API)
!pip install -q transformers accelerate sentencepiece ftfy==6.1.1 timm safetensors

import torch
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import os, sys

# Model to use (good balance of speed and quality)
MODEL_NAME = "nlpconnect/vit-gpt2-image-captioning"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load model (this downloads weights the first time)
cap_model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Caption model loaded:", MODEL_NAME)

In [ ]:
# Local caption helper (call this from predict_with_caption)
import numpy as np

def get_caption_local(pil_img, max_length=32, num_beams=4):
    """
    Generates a one-sentence caption for pil_img using the local model loaded above.
    """
    try:
        if pil_img.mode != "RGB":
            pil_img = pil_img.convert("RGB")
        pixel_values = processor(images=pil_img, return_tensors="pt").pixel_values
        pixel_values = pixel_values.to(device)
        output_ids = cap_model.generate(pixel_values, max_length=max_length, num_beams=num_beams)
        caption = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        return caption
    except Exception as e:
        return f"[Local caption error: {e}]"

In [ ]:
import numpy as np

CONF_THRESHOLD = 0.55  # optional, tweakable

def predict_with_caption(img):
    """
    Minimal replacement that returns (prob_dict, text_output).
    Uses your existing `model` and `class_names` for classification and
    `get_caption_local` for generating the caption with the local model.
    """
    if img is None:
        return {}, "No image provided."

    # ensure we have a PIL RGB image and create a resized copy for classifier
    try:
        pil_full = img.convert("RGB")
    except Exception:
        from PIL import Image
        pil_full = Image.fromarray(img).convert("RGB")

    pil_for_model = pil_full.resize((IMG_SIZE, IMG_SIZE))

    # classifier (unchanged behavior)
    try:
        x = np.array(pil_for_model).astype(np.float32) / 255.0
        x = np.expand_dims(x, axis=0)
        preds = model.predict(x)[0]
        best_idx = int(np.argmax(preds))
        best_class = class_names[best_idx]
        confidence = float(preds[best_idx])
        prob_dict = {class_names[i]: float(preds[i]) for i in range(len(class_names))}
    except Exception as e:
        print("Classifier error in predict_with_caption:", e)
        prob_dict = {}
        best_class = "Unknown"
        confidence = 0.0

    # caption (local model) — uses full-size image for better results
    try:
        # call the local captioning function you added earlier
        caption = get_caption_local(pil_full)
    except Exception as e:
        caption = f"[Local caption error: {e}]"

    # optional low-confidence notice
    if confidence < CONF_THRESHOLD:
        caption = "Note: model not confident about category. " + caption

    text_output = (
        f"Predicted Category: {best_class}\n"
        f"Confidence: {confidence * 100:.2f}%\n\n"
        f"Caption: {caption}"
    )

    return prob_dict, text_output

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict_with_caption,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=[
        gr.Label(label="Prediction probabilities"),
        gr.Textbox(label="Result (category, confidence & caption)", lines=5)
    ],
    title="AI Image Caption Generator",
    description="Upload an image. The model predicts its category and shows a caption from that category."
)

demo.launch()